# 4. 싱글턴 vs 멀티턴

**목표** — LLM API가 이전 대화를 기억하지 않는다는 것을 직접 확인하고, 그럼에도 대화가 이어지게 만드는 방법과 **그 대가(비용)** 를 이해한다.

**소요 시간** 약 80분

| 다루는 것 | |
| --- | --- |
| 1 | 싱글턴 — 기억하지 못한다 |
| 2 | 멀티턴 A — 메시지 배열 직접 관리 |
| 3 | 토큰이 어떻게 불어나는가 |
| 4 | 멀티턴 B — 세션 API |
| 5 | 비용 폭증 대응: 슬라이딩 윈도우 |
| 6 | 연습문제 |

> 오늘 배운 것이 여기서 전부 합쳐진다. **토큰(02) → 길이 제어(03) → 대화 누적(04)**

## 0. 준비

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from google import genai
from openai import OpenAI

load_dotenv(override=True)

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
oa = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gemini-3.1-flash-lite"
OA_MODEL = "gpt-4o-mini"

print("준비 완료")

준비 완료


## 1. 싱글턴 — 서버는 기억하지 않는다

두 번 연달아 호출한다. 두 번째 호출에서 **앞에서 알려준 정보를 기억하는지** 본다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
r1 = oa.responses.create(model=OA_MODEL, input="내 이름은 김철수야. 기억해줘.")
print("1턴 →", r1.output_text.strip())

r2 = oa.responses.create(model=OA_MODEL, input="내 이름이 뭐라고 했지?")
print("2턴 →", r2.output_text.strip())
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 2턴에서 이름을 기억하지 못하는지 본다

In [2]:
r1 = oa.responses.create(model=OA_MODEL, input="내 이름은 김철수야. 기억해줘.")
print("1턴 →", r1.output_text.strip())

r2 = oa.responses.create(model=OA_MODEL, input="내 이름이 뭐라고 했지?")
print("2턴 →", r2.output_text.strip())

1턴 → 안녕하세요, 김철수님! 반갑습니다. 도움이 필요하시면 언제든지 말씀해 주세요!
2턴 → 죄송하지만, 이전 대화를 기억할 수 없어서 이름을 알지 못해요. 당신의 이름이 궁금하다면 알려주시면 좋겠습니다!


### 기억하지 못한다

두 호출은 **완전히 독립적**이다. 서버 입장에서 2번째 요청은 처음 보는 사람의 첫 질문이다.

> ChatGPT 화면에서 대화가 이어지는 것처럼 보이는 건, **화면 쪽에서 이전 대화를 매번 통째로 다시 보내주고 있기 때문**이다.

## 2. 멀티턴 A — 메시지 배열 직접 관리

기억하게 하려면 **이전 대화를 우리가 다시 넣어서 보내면 된다.**
이때 모델이 이전에 한 말은 `assistant` 역할로 넣는다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
messages = [
    {"role": "user", "content": "내 이름은 김철수야. 기억해줘."},
    {"role": "assistant", "content": r1.output_text},   # 모델이 아까 한 답변을 되돌려준다
    {"role": "user", "content": "내 이름이 뭐라고 했지?"},
]

r3 = oa.responses.create(model=OA_MODEL, input=messages)
print(r3.output_text.strip())
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 이번에는 이름을 기억하는지 본다

In [4]:
messages = [
    {"role": "user", "content": "내 이름은 홍길동이야. 기억해줘."},
    {"role": "assistant", "content": r1.output_text},   # 모델이 아까 한 답변을 되돌려준다
    {"role": "user", "content": "내 이름이 뭐라고 했지?"},
]

r3 = oa.responses.create(model=OA_MODEL, input=messages)
print(r3.output_text.strip())

당신의 이름은 홍길동이라고 하셨습니다. 기억하고 있습니다!


In [6]:
r3 = oa.responses.create(model=OA_MODEL, input="내 이름은 홍길동이야. 꼭 기억해줘.")
r3.output_text.strip()

'안녕하세요, 홍길동님! 기억하겠습니다. 어떤 이야기를 나눠보고 싶으신가요?'

이번엔 기억한다. **모델이 똑똑해진 게 아니라, 우리가 답을 프롬프트에 같이 넣어준 것뿐이다.**

여기서 오늘의 핵심 결론이 나온다.

> ### 멀티턴 = 매 턴마다 이전 대화 전체를 다시 보내는 것
> 그래서 **대화가 길어질수록 입력 토큰이 계속 불어난다.**

## 3. 토큰이 어떻게 불어나는가

5턴짜리 대화를 진행하면서, **매 턴의 입력 토큰을 기록**한다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
history = []
log = []

questions = [
    "파이썬을 배우고 있어. 내 이름은 김철수야.",
    "리스트와 튜플의 차이가 뭐야?",
    "그럼 딕셔너리는?",
    "방금 설명한 3가지 중에 수정이 가능한 건 뭐야?",
    "내 이름이 뭐라고 했지?",
]

for turn, q in enumerate(questions, start=1):
    history.append({"role": "user", "content": q})

    r = oa.responses.create(
        model=OA_MODEL,
        input=history,
        max_output_tokens=150,
    )
    answer = r.output_text.strip()
    history.append({"role": "assistant", "content": answer})

    log.append({
        "턴": turn,
        "질문": q[:22] + "...",
        "입력토큰": r.usage.input_tokens,
        "출력토큰": r.usage.output_tokens,
    })
    print(f"[{turn}턴] {answer[:70]}...")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 턴이 늘수록 입력 토큰이 커지는지 본다

In [7]:
history = []
log = []

questions = [
    "파이썬을 배우고 있어. 내 이름은 김철수야.",
    "리스트와 튜플의 차이가 뭐야?",
    "그럼 딕셔너리는?",
    "방금 설명한 3가지 중에 수정이 가능한 건 뭐야?",
    "내 이름이 뭐라고 했지?",
]

for turn, q in enumerate(questions, start=1):
    history.append({"role": "user", "content": q})

    r = oa.responses.create(
        model=OA_MODEL,
        input=history,
        max_output_tokens=150,
    )
    answer = r.output_text.strip()
    history.append({"role": "assistant", "content": answer})

    log.append({
        "턴": turn,
        "질문": q[:22] + "...",
        "입력토큰": r.usage.input_tokens,
        "출력토큰": r.usage.output_tokens,
    })
    print(f"[{turn}턴] {answer[:70]}...")

[1턴] 안녕하세요, 김철수님! 파이썬을 배우고 계신다니 멋지네요. 파이썬에 대해 궁금한 점이나 도움이 필요한 부분이 있으면 언제든지 ...
[2턴] 리스트와 튜플은 파이썬에서 데이터를 저장하는 데 사용되는 두 가지 기본적인 자료형입니다. 이 두 가지의 주요 차이점은 다음과 ...
[3턴] 딕셔너리(Dict)는 파이썬에서 키-값 쌍을 저장하는 데 사용되는 자료형입니다. 리스트와 튜플과는 다른 특징이 있습니다. 아래...
[4턴] 설명한 세 가지 자료형 중에서 수정이 가능한(mutable) 것은 **리스트**와 **딕셔너리**입니다. 

1. **리스트 ...
[5턴] 당신의 이름은 김철수입니다!...


**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
df = pd.DataFrame(log)
df["누적토큰"] = (df["입력토큰"] + df["출력토큰"]).cumsum()
df["입력토큰 그래프"] = df["입력토큰"].apply(lambda n: "█" * (n // 40))
df
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 입력토큰 그래프가 계단식으로 길어지는지 본다

In [8]:
df = pd.DataFrame(log)
df["누적토큰"] = (df["입력토큰"] + df["출력토큰"]).cumsum()
df["입력토큰 그래프"] = df["입력토큰"].apply(lambda n: "█" * (n // 40))
df

,턴,질문,입력토큰,출력토큰,누적토큰,입력토큰 그래프
0,1,파이썬을 배우고 있어. 내 이름은 김철수...,24,60,84,
1,2,리스트와 튜플의 차이가 뭐야?...,103,150,337,██
2,3,그럼 딕셔너리는?...,269,150,756,██████
3,4,방금 설명한 3가지 중에 수정이 가능한 ...,443,150,1349,███████████
4,5,내 이름이 뭐라고 했지?...,609,11,1969,███████████████


### 관찰

- **입력 토큰이 턴마다 계단식으로 늘어난다.** 5턴째 질문("내 이름이 뭐라고 했지?")은 매우 짧지만 입력 토큰은 가장 크다.
- 마지막 턴에서 이름을 정확히 기억한다 — 1턴의 내용이 계속 같이 전송되고 있기 때문이다.
- **누적 토큰은 턴 수에 대해 제곱에 가깝게 증가한다.** 매 턴 전체 이력을 다시 보내기 때문이다.

이게 실무에서 문제가 되는 지점이다.

| 문제 | 결과 |
| --- | --- |
| 비용 | 대화가 길수록 턴당 비용이 계속 오른다 |
| 속도 | 입력이 길수록 응답이 느려진다 |
| 한계 | 컨텍스트 윈도우를 넘으면 아예 호출이 실패한다 |

## 4. 멀티턴 B — 세션 API

이력 관리를 SDK가 대신 해주는 방식도 있다.

### Gemini — `client.chats`

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
chat = client.chats.create(model=MODEL)

print("1턴 →", chat.send_message("내 이름은 김철수야.").text.strip())
print("2턴 →", chat.send_message("파이썬 리스트가 뭐야? 한 문장으로.").text.strip())
print("3턴 →", chat.send_message("내 이름이 뭐라고 했지?").text.strip())

print()
print("--- 저장된 대화 이력 ---")
for msg in chat.get_history():
    print(f"[{msg.role}] {msg.parts[0].text.strip()[:50]}")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 이력의 역할이 user / model 로 나오는지 본다

`chat` 객체가 이력을 들고 있다가 매번 같이 보낸다. **내부적으로 하는 일은 2절과 완전히 같다** — 편의를 위해 감싼 것뿐이다.

> **참고: 역할 이름이 다르다.** 출력된 이력의 역할이 `user` / **`model`** 로 나온다. OpenAI는 같은 것을 `assistant`라고 부른다.
> 개념은 동일하지만 이름이 다르므로, 두 SDK를 오갈 때 변환이 필요하다. (`3_chat-service`에서 DB에 저장할 때는 `assistant`로 통일한다)

### OpenAI — `previous_response_id`

OpenAI는 응답을 서버에 저장해두고, **이전 응답의 id만 넘겨서** 대화를 이어갈 수 있다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
a = oa.responses.create(model=OA_MODEL, input="내 이름은 김철수야.")
print("1턴 →", a.output_text.strip())

b = oa.responses.create(
    model=OA_MODEL,
    input="내 이름이 뭐라고 했지?",
    previous_response_id=a.id,      # 이전 대화를 서버에서 이어붙인다
)
print("2턴 →", b.output_text.strip())

print()
print("2턴 입력 토큰:", b.usage.input_tokens, "← 이력이 서버에서 붙었으므로 짧은 질문인데도 크다")
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 짧은 질문인데 입력 토큰이 큰지 본다

### 두 방식 비교

| | 직접 관리 (2절) | 세션 API (4절) |
| --- | --- | --- |
| 코드량 | 많다 | 적다 |
| 이력 제어 | **완전히 자유** (자르기·요약·수정 가능) | 제한적 |
| 이력 저장 위치 | 내 프로그램 | SDK 객체 또는 제공자 서버 |
| DB에 저장하기 | **쉽다** — 이미 내 손에 있다 | 별도로 꺼내야 한다 |
| 비용 | 동일 | 동일 (전송량은 결국 같다) |

> **주의: 어느 쪽을 쓰든 토큰이 누적되는 건 똑같다.** 세션 API는 코드를 줄여줄 뿐, 비용을 줄여주지 않는다.
>
> 실제 서비스는 대화를 **DB에 저장**해야 하므로(새로고침해도 남아있어야 하니까) **직접 관리 방식**을 쓰는 경우가 많다. `3_chat-service`에서 그렇게 만든다.

## 5. 비용 폭증 대응 — 슬라이딩 윈도우

가장 단순하고 효과적인 방법은 **최근 N턴만 보내는 것**이다.

**참고 코드** — 아래를 보고 **다음 셀에 직접 입력**한 뒤 실행한다.

```python
def trim(history, keep_turns=2):
    """최근 keep_turns 턴(=user/assistant 쌍)만 남긴다."""
    return history[-(keep_turns * 2):]


full = history                       # 3절에서 쌓은 5턴 전체
trimmed = trim(history, keep_turns=2)

q = {"role": "user", "content": "지금까지 설명한 걸 한 문장으로 요약해줘."}

r_full = oa.responses.create(model=OA_MODEL, input=full + [q], max_output_tokens=150)
r_trim = oa.responses.create(model=OA_MODEL, input=trimmed + [q], max_output_tokens=150)

print(f"전체 이력  입력토큰: {r_full.usage.input_tokens:5d}")
print(f"최근 2턴만 입력토큰: {r_trim.usage.input_tokens:5d}")
print(f"→ 입력 토큰 {100 * (1 - r_trim.usage.input_tokens / r_full.usage.input_tokens):.1f}% 절감")
print()
print("[전체] ", r_full.output_text.strip()[:100])
print("[2턴] ", r_trim.output_text.strip()[:100])
```

In [ ]:
# TODO: 위 참고 코드를 직접 입력하고 실행한다
#       확인: 최근 2턴만 보내면 입력 토큰이 얼마나 줄어드는지 본다

### 트레이드오프

토큰은 줄었지만 **잘라낸 부분의 정보는 잃는다.** 위에서 이름을 물어보면 최근 2턴만 남긴 쪽은 답하지 못한다.

실무에서는 상황에 맞게 조합한다.

| 전략 | 방법 | 특징 |
| --- | --- | --- |
| **슬라이딩 윈도우** | 최근 N턴만 전송 | 간단, 오래된 정보 손실 |
| **요약 압축** | 오래된 대화를 LLM으로 요약해 한 덩어리로 치환 | 정보 보존, 요약 호출 비용 추가 |
| **핵심 정보 고정** | 이름·설정 등은 `system`에 박아두고 나머지만 자른다 | 실무에서 가장 많이 쓴다 |

## 6. 연습문제

### 연습 4-1. 대화 비용 계산기

3절에서 만든 `log`를 이용해, **5턴 대화 전체의 총비용**을 계산한다.
그리고 이 대화가 **20턴까지 이어졌다면** 비용이 얼마가 될지 추정해본다.

In [ ]:
# TODO: 노트북 02에서 확인한 단가를 채운다
PRICE_IN_PER_1M = None
PRICE_OUT_PER_1M = None
USD_KRW = None


def cost(input_tokens, output_tokens):
    usd = (input_tokens / 1_000_000) * PRICE_IN_PER_1M + (output_tokens / 1_000_000) * PRICE_OUT_PER_1M
    return {"usd": usd, "krw": usd * USD_KRW}


# TODO: df의 입력토큰/출력토큰 합계로 5턴 대화의 총비용을 계산한다

# TODO: 턴이 늘수록 입력 토큰이 어떻게 증가했는지 보고, 20턴이면 얼마일지 추정한다
#       (힌트: df["입력토큰"]의 턴당 증가량을 보고 외삽한다)

### 연습 4-2. 핵심 정보를 지키면서 이력 줄이기

5절의 `trim()`은 이름 같은 중요한 정보까지 잘라버린다.
**`system` 지침에 핵심 정보를 고정**하고 나머지 이력만 자르는 방식으로 개선해서,
**최근 2턴만 남겨도 이름을 기억하게** 만들어본다.

In [ ]:
# TODO: 핵심 정보를 담은 system 지침을 만든다 (예: 사용자 이름)
system_note = ""

# TODO: trim()으로 최근 2턴만 남긴 이력 + system 지침으로 호출한다
#       (OpenAI Responses API에서는 instructions= 로 system 지침을 넘긴다)

# TODO: "내 이름이 뭐라고 했지?"에 제대로 답하는지 확인하고,
#       입력 토큰이 전체 이력을 보낼 때보다 얼마나 적은지 비교한다

## 정리

- [ ] LLM API가 이전 대화를 기억하지 않는다는 것을 직접 확인했다
- [ ] `assistant` 역할로 이전 답변을 되돌려주면 대화가 이어진다는 것을 안다
- [ ] 턴이 늘수록 입력 토큰이 계단식으로 증가하는 것을 관찰했다
- [ ] 세션 API가 비용을 줄여주지는 않는다는 것을 안다
- [ ] 슬라이딩 윈도우의 효과와 트레이드오프를 설명할 수 있다

---

## 오늘 배운 것이 어디로 이어지는가

지금 이 대화 이력은 **노트북을 닫으면 사라진다.** 실제 서비스라면 새로고침해도 남아있어야 한다.

**다음** → [05_fastapi_integration.ipynb](./05_fastapi_integration.ipynb)
오늘의 마지막 조각이다. 지금까지 노트북 안에서만 돌던 호출을 **FastAPI 엔드포인트**로 만든다.

> **대화를 어디에 저장할 것인가** → 이건 다음 과정(11일차 `1_supabase-basic-test` → 12일차 `3_chat-service`)의 주제다.
> `conversations` / `messages` 테이블을 설계하고, 오늘 만든 `history` 리스트를 DB에 넣는다.

수고했다.